# Demonstração ao vivo: a folha escrita à caneta, lida pelos dois OCRs do projeto

Alguém escreve numa folha, a folha é fotografada com o celular e a foto é lida pelos **dois OCRs do projeto**, sempre os mesmos:

- **pipeline**: o pipeline final (`configs/final_v6.json`), local, sem LLM. A foto é apenas reduzida à escala do pipeline (a das páginas do livro, ~650 × 880 px) e entra na mesma configuração que transcreve o livro, sem nenhum ajuste.
- **LLM**: o melhor LLM da avaliação do projeto (`Transcription_Evaluation.ipynb`): **Gemini 3.8 Flash**, esforço de raciocínio `low`, sobre a própria foto, pelo motor `ocr_historico/motor_llm.py` (OpenRouter). **Cada foto custa dinheiro**: nada é enviado enquanto `PERMITIR_GASTO = False` (a requisição é montada e gravada, mas não sai da máquina).

O cartão mostra a foto passando por cada etapa (a redução, depois as operações de imagem do pipeline e a segmentação) e as duas leituras lado a lado.

- **No VS Code:** abra este caderno na raiz do repositório com o kernel do `.venv`, conecte o celular pelo cabo e copie a foto para `dados/demo/`.
- **No Colab:** abra pelo GitHub (**Arquivo → Abrir notebook → GitHub**, ramo `demo-ao-vivo`), escolha um ambiente com GPU e rode as células em ordem; a foto entra pela célula de upload e o cartão é baixado para o seu computador.

Roteiro completo e dicas de escrita: `docs/DEMO.md`. A lógica está em `ocr_pipeline/demo.py`.

## Ambiente

In [ ]:
# No Colab: clona o repositório, instala o Kraken e as dependências (o torch do Colab é mantido) e confere os modelos.
# No VS Code: abra o caderno na raiz do repositório com o kernel do .venv; esta célula só confere a pasta.
import os
import subprocess
import sys
from pathlib import Path

REPOSITORIO = 'https://github.com/fabiolobato/hackathon_ocr_historico.git'
RAMO = 'demo-ao-vivo'  # o ramo do GitHub que tem a demo
NO_COLAB = 'google.colab' in sys.modules

if NO_COLAB and not Path('ocr_pipeline/demo.py').is_file():
    clone = Path('/content/hackathon_ocr_historico')
    if not clone.is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '-b', RAMO, REPOSITORIO, str(clone)], check=True)
    elif not (clone / 'ocr_pipeline/demo.py').is_file():  # clone antigo: pega o que entrou no ramo depois dele
        subprocess.run(['git', 'pull', '--ff-only'], cwd=clone, check=False)
    if not (clone / 'ocr_pipeline/demo.py').is_file():
        raise RuntimeError(f'o ramo {RAMO} de {REPOSITORIO} ainda não tem a demo (ocr_pipeline/demo.py): é preciso '
                           'fazer commit e push dos arquivos da demo nesse repositório, ou apontar REPOSITORIO/RAMO '
                           'acima para o repositório e o ramo que já os têm.')
    subprocess.run([sys.executable, 'setup_pipeline.py'], cwd=clone / 'entrega_ocr', check=True)  # Kraken + dependências
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow-heif'], check=True)  # fotos HEIC do iPhone
    os.chdir(clone)
if not Path('ocr_pipeline/demo.py').is_file():
    raise RuntimeError(f'abra o caderno na raiz do repositório (pasta atual: {Path.cwd()})')
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))
print('pasta de trabalho:', Path.cwd())

## Os dois OCRs

Com `USAR_LLM = True`, o Gemini 3.8 Flash lê a mesma foto ao lado do pipeline (`False` roda só o pipeline). `PERMITIR_GASTO = False` o mantém em simulação: a requisição é montada e gravada, mas nada é enviado e nada é cobrado. Mude `PERMITIR_GASTO` para `True` só na apresentação, com a chave do OpenRouter definida (`OPENROUTER_API_KEY` no ambiente, segredo `API_KEY` no Colab ou um `.env` na raiz): com `True`, esta célula já falha se a chave não existir, para não descobrir isso no meio da apresentação. Ela também carrega os modelos do pipeline (10 a 20 s).

In [ ]:
import importlib
import logging

import torch

import ocr_pipeline.demo
importlib.reload(ocr_pipeline.demo)  # um kernel já aberto usaria a versão antiga do módulo
from ocr_pipeline.demo import LLM_EXAMPLE_TEXT, LLM_MODEL, PHOTOS, PIPELINE_CONFIG, PREPARACAO, build_demo, latest_photo

print('módulo:', ocr_pipeline.demo.__file__)
print('preparação da foto:', PREPARACAO)

USAR_LLM = True           # True: o Gemini 3.8 Flash lê a mesma foto, lado a lado com o pipeline; False: só o pipeline
PERMITIR_GASTO = True     # True: envia as fotos ao OpenRouter (pago, centavos por foto); False: o LLM só simula
EXEMPLO_LLM = False       # True (só com PERMITIR_GASTO = False): ensaio da página; o cartão do LLM mostra um texto de exemplo, nada é enviado
DISPOSITIVO = 'cuda:0' if torch.cuda.is_available() else 'cpu'

logging.basicConfig(level=logging.WARNING)
demo = build_demo(device=DISPOSITIVO, allow_spend=PERMITIR_GASTO, use_llm=USAR_LLM, open_cards=True,  # o cartão abre no navegador
                  llm_example=LLM_EXAMPLE_TEXT if EXEMPLO_LLM else None)
demo.warm_up()
PHOTOS.mkdir(parents=True, exist_ok=True)
modo = 'envio autorizado' if PERMITIR_GASTO else 'exemplo, nada enviado' if EXEMPLO_LLM else 'simulação'
llm = f' + {LLM_MODEL} ({modo})' if USAR_LLM else ''
print(f'OCRs: {PIPELINE_CONFIG}{llm}')
print('dispositivo:', DISPOSITIVO)
print('pasta das fotos:', PHOTOS.resolve())
print('abertura do cartão:', f'página {demo.book.number} do livro' if demo.book else 'sem a página do livro')

## A foto

Fotografe a folha de cima, **preenchendo o quadro** (nada é recortado: a foto inteira é reduzida ao tamanho de uma página do livro), sem a sombra do celular sobre o texto.

- **VS Code:** conecte o celular pelo cabo (no Android, escolha "transferência de arquivos"), abra-o no Explorador de Arquivos (`DCIM\Camera`) e arraste a foto para `dados/demo/`. As fotos usadas ficam guardadas aí, dentro do projeto.
- **Colab:** rode a célula abaixo e escolha a foto.

In [ ]:
# No Colab: abre o seletor de arquivos e guarda o que for enviado em dados/demo/. No VS Code: não faz nada.
if NO_COLAB:
    from google.colab import files

    for nome, dados in files.upload().items():
        (PHOTOS / nome).write_bytes(dados)
        print('recebida:', PHOTOS / nome)
else:
    print('copie a foto do celular (cabo) para', PHOTOS.resolve())

## Ler a foto

Rode esta célula a cada foto nova. Por padrão ela pega a foto mais recente de `dados/demo/`; `FOTO` aponta para uma específica. As leituras aparecem aqui embaixo e o **cartão completo abre no navegador** (ele também fica salvo em `outputs/demo/cartoes/`). No Colab o cartão é **baixado** para o seu computador: abra o arquivo `.html` no navegador e deixe essa aba no projetor.

In [ ]:
FOTO = None            # None: a foto mais recente de dados/demo/. Ou um caminho: Path('dados/demo/IMG_1234.jpg')

foto = Path(FOTO) if FOTO else latest_photo()
print('foto:', foto)
resultado = demo.run(foto)
for leitura in resultado.readings:
    print(f'\n=== {leitura.title} · {leitura.details} ===\n{leitura.text or leitura.error}')
if NO_COLAB:  # o navegador do Colab não abre arquivos da máquina virtual: o cartão desce para o seu computador
    from google.colab import files
    files.download(str(resultado.card_file))
    print('\ncartão baixado; abra-o no navegador:', resultado.card_file.name)
else:
    print('\ncartão aberto no navegador:', resultado.card_file)

## Onde fica tudo

As fotos ficam em `dados/demo/`, dentro do projeto. O resto vai para `outputs/demo/` (fora do git): `entrada/` (o que cada OCR recebeu: `<id>.pipeline.jpg`, `<id>.llm.png`), `textos/` (`<id>.pipeline.txt`, `<id>.llm.txt`), `cartoes/` (`<id>.html`, o mesmo cartão em página avulsa), `llm/` (requisição e resposta de cada foto, sem a imagem) e `cache/`. Cada foto recebe um nome único, então o cache nunca confunde duas fotos.

O mesmo fluxo roda pela linha de comando: `python -m ocr_pipeline.demo --vigiar --abrir` abre um cartão no navegador a cada foto nova em `dados/demo/`.